# Tecan 브링업 프로브 — 셀 4개, 출력은 그대로 복사해 세션에 붙이기

위→아래 순서로 실행. 각 셀이 `STEP`/`VERDICT` 라인을 찍는다 — **출력 전체를 복사해 붙여주면 바로 판정한다.**
⛔ read-only 상태쿼리(`Q`·`?`)만 존재. 모션·NVM 명령 없음. 정비 툴(localhost:8000)의 [자동 연결]은 끄고 실행.

In [1]:
# STEP 1 — 환경·장치 파악 (맥/파이 공통)
import platform, subprocess, sys

print(f"STEP=1 os={platform.system()} py={sys.version.split()[0]}")
for mod in ("serial", "usb", "libusb_package"):
    try:
        __import__(mod); print(f"dep {mod}: OK")
    except ImportError:
        print(f"dep {mod}: MISSING  → pip install pyserial pyusb libusb-package")

import serial.tools.list_ports as lp
EXCLUDE = ("bluetooth", "debug-console", "wlan", "buds")
cands = [p for p in lp.comports() if not any(h in p.device.lower() for h in EXCLUDE)]
for p in cands:
    print(f"serial-candidate {p.device} vid:pid={p.vid}:{p.pid}")

usb_serial_chip = None
if platform.system() == "Darwin":
    tree = subprocess.run(["system_profiler", "SPUSBDataType"], capture_output=True, text=True).stdout.lower()
    for vid, chip in (("0x067b", "PL2303"), ("0x1a86", "CH340"), ("0x10c4", "CP210x"), ("0x0403", "FTDI")):
        if vid in tree:
            usb_serial_chip = chip
            print(f"usb-tree chip={chip} vid={vid}")

if cands:
    print("VERDICT: KERNEL_PATH_READY —", cands[0].device)
elif usb_serial_chip:
    print(f"VERDICT: USERSPACE_PATH_READY — {usb_serial_chip} (드라이버 없음, pyusb 로 진행)")
else:
    print("VERDICT: NO_ADAPTER — USB 케이블/허브 물리 연결 확인")

STEP=1 os=Darwin py=3.13.5
dep serial: OK
dep usb: OK
dep libusb_package: OK
usb-tree chip=PL2303 vid=0x067b
VERDICT: USERSPACE_PATH_READY — PL2303 (드라이버 없음, pyusb 로 진행)


In [1]:
# STEP 2 — 포트 열기 (커널 경로 우선 → 유저스페이스 자동 폴백)
from pl2303py import open_pump_serial

ser = open_pump_serial(baudrate=9600, timeout=0.5)
print(f"VERDICT: OPEN_OK path={ser.name}")

[open_pump_serial] /dev 노드 없음 → 유저스페이스(pyusb) 경로
VERDICT: OPEN_OK path=pyusb:067b:23c3


In [1]:
# STEP 3 — 프로브: Q(테칸 정규) + ?(호환) × 주소 1~3.  ⚠️ 펌프 24V 전원 먼저.
import time
ETX = 0x03

def raw_probe(ser, addr, cmd, attempts=5, deadline_s=6.0):
    end = time.monotonic() + deadline_s
    for _ in range(attempts):
        if time.monotonic() > end:
            break
        ser.reset_input_buffer()
        ser.write(f"/{addr}{cmd}R\r".encode())
        buf, t0 = b"", time.monotonic()
        while time.monotonic() - t0 < 1.0:
            chunk = ser.read(64)
            if chunk:
                buf += chunk
                if bytes([ETX]) in buf:
                    return buf
        if buf:
            return buf
        time.sleep(0.2)
    return b""

responding = []
for cmd in ("Q", "?"):
    for addr in (1, 2, 3):
        rx = raw_probe(ser, addr, cmd)
        print(f"probe cmd={cmd} addr={addr} rx={rx.hex(' ') or 'SILENT'}")
        if rx:
            responding.append((cmd, addr))

if responding:
    print(f"VERDICT: RESPONDING {responding}")
else:
    print("VERDICT: ALL_SILENT — STEP 4(baud 스윕) 실행")

probe cmd=Q addr=1 rx=2f 30 60 03
probe cmd=Q addr=2 rx=SILENT
probe cmd=Q addr=3 rx=SILENT
probe cmd=? addr=1 rx=2f 30 60 30 03
probe cmd=? addr=2 rx=SILENT
probe cmd=? addr=3 rx=SILENT
VERDICT: RESPONDING [('Q', 1), ('?', 1)]


In [ ]:
# STEP 4 — baud 스윕 (STEP 3 이 ALL_SILENT 일 때만)
from pl2303py import open_pump_serial

for baud in (9600, 19200, 38400, 115200):
    ser.close()
    ser = open_pump_serial(baudrate=baud, timeout=0.5)
    rx = raw_probe(ser, 1, "Q", attempts=3, deadline_s=3.0)
    print(f"baud={baud} rx={rx.hex(' ') or 'SILENT'}")
print("VERDICT: (응답 나온 baud 를 세션에 보고)")

## 판정 규칙 (참고 — 세션에 출력 붙이면 이대로 판정한다)
- STEP3 `Q` 응답 → 정상, 프로덕션 어댑터 그대로 사용 가능.
- `Q` SILENT + `?` 응답 → 어댑터 status_cmd 이슈(재기동 루프 조건) — 코드 수정 필요, 즉시 보고.
- ALL_SILENT + STEP4 도 침묵 → RS485 A/B 배선·24V 전원·DIP 주소 층 (소프트웨어 무죄).
- STEP4 특정 baud 에서만 응답 → 펌프 통신 설정이 9600 아님 — 통일 여부 결정.